# Stage 5: Association Rule Mining — State-Level Syllabus Demonstration
**Project**: Cyber Crime Analytics for National Security  
**Data Source**: National Crime Records Bureau (NCRB) 2023 Master Dataset (`master_state_2023.csv`)  
**Methodological Position**: State-Level Profile Association Mining (Demonstration of Syllabus Concepts)

---

## 1. Methodological Context & Scope

### Important Clarification: Aggregate State Observations vs. Incident Transactions
This dataset is **not** an incident-level transactional database. In retail market basket analysis or cyber incident logs, a transaction represents a single event:
$$\text{Transaction } t_i \to \{\text{Item } A, \text{Item } B, \text{Item } C, \dots\}$$
In this official NCRB dataset, observations represent **aggregate annual totals for 36 Indian States and Union Territories (UTs)** ($n = 36$).

### Pedagogical & Analytical Objective
This analysis demonstrates the core concepts of Data Mining Association Analysis:
1. **Transaction Matrix Construction** from continuous regional profiles using objective percentile thresholds (median split).
2. **Itemset Identification & Support Counting** ($k / 36$ states).
3. **Apriori Algorithm Execution** to find frequent itemsets without combinatorial explosion.
4. **Association Rule Generation & Filtering** using Support, Confidence, and Lift.
5. **Academic Interpretation & Non-Causal Guardrails**.

> **Crucial Guardrail**: Association rules discovered here represent **descriptive state-level profile co-occurrences** across 36 regional observations. They **do not** prove incident-level behavioral co-occurrence and **must never be interpreted as causal relationships or population-level behavioral laws**.


In [1]:
# Setup environment and imports
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import Stage 5 module
from src.association_rules import (
    build_state_transaction_matrix,
    mine_frequent_itemsets,
    mine_association_rules,
    plot_frequent_itemset_support,
    plot_association_rules_scatter,
    export_association_tables,
    ITEM_DEFINITIONS
)

print(f"Working Directory: {project_root}")
print("Association Rule Mining module loaded successfully.")


Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics
Association Rule Mining module loaded successfully.


---
## Section B — Transaction Design & Threshold Selection

### Item Design Principles
To prevent trivial rules, structural duplication, and volume leakage:
1. **No Total Cases Item**: `total_cases` is omitted so rules do not merely reflect overall state scale.
2. **No Parent/Subtotal Categories**: Only independent leaf categories, major motives, specific subset totals, and legal shares are used.
3. **Objective Median Split Threshold**: Each continuous metric is dichotomized at its **50th percentile (median)** across all 36 States/UTs. An item is `True` if a state is in the upper 50% for that characteristic.


In [2]:
# Load validated 2023 master dataset
data_path = project_root / 'data' / 'processed' / 'master_state_2023.csv'
master_df = pd.read_csv(data_path)
print(f"Loaded master dataset with shape: {master_df.shape} (36 States/UTs)")

# Build binary transaction matrix
transaction_matrix, threshold_summary = build_state_transaction_matrix(master_df, threshold_percentile=0.50)

print("\n--- Item Threshold & Frequency Summary ---")
display(threshold_summary[['item_name', 'category', 'cutoff_value', 'active_states_count', 'active_states_pct', 'description']])


Loaded master dataset with shape: (36, 164) (36 States/UTs)

--- Item Threshold & Frequency Summary ---


,item_name,category,cutoff_value,active_states_count,active_states_pct,description
0,HIGH_FRAUD_MOTIVE,Motive,119.5000,18,50.00,Fraud motive count >= median threshold across States/UTs
1,HIGH_SEC66D_CHEATING,IT Act Category,37.5000,18,50.00,Sec. 66D Personation Cheating count >= median threshold
2,HIGH_IDENTITY_THEFT,IT Act Category,8.0000,19,52.78,Sec. 66C Identity Theft count >= median threshold
3,HIGH_IT_ACT_SHARE,Legal Composition,0.6875,18,50.00,Share of cases under IT Act >= median threshold
4,HIGH_EXTORTION_MOTIVE,Motive,9.5000,18,50.00,Extortion motive count >= median threshold
5,HIGH_SEXUAL_EXPLOITATION_MOTIVE,Motive,30.5000,18,50.00,Sexual Exploitation motive count >= median threshold
6,HIGH_WOMEN_CYBERCRIME,Vulnerable Groups,131.0000,18,50.00,Total cybercrimes against women >= median threshold
7,HIGH_CHILD_CYBERCRIME,Vulnerable Groups,12.0000,18,50.00,Total cybercrimes against children >= median threshold


In [3]:
# Inspect constructed State/UT Transaction Matrix (Sample of Top & Bottom States)
print(f"Transaction Matrix Shape: {transaction_matrix.shape} (36 Transactions x {transaction_matrix.shape[1]} Items)")
display(transaction_matrix.head(10))


Transaction Matrix Shape: (36, 8) (36 Transactions x 8 Items)


,HIGH_FRAUD_MOTIVE,HIGH_SEC66D_CHEATING,HIGH_IDENTITY_THEFT,HIGH_IT_ACT_SHARE,HIGH_EXTORTION_MOTIVE,HIGH_SEXUAL_EXPLOITATION_MOTIVE,HIGH_WOMEN_CYBERCRIME,HIGH_CHILD_CYBERCRIME
state_name,,,,,,,,
Andhra Pradesh,True,True,True,False,True,True,True,True
Arunachal Pradesh,False,False,True,True,False,False,False,False
Assam,True,True,True,True,True,True,True,True
Bihar,True,True,True,False,True,True,True,False
Chhattisgarh,False,False,False,False,False,True,True,True
Goa,False,True,False,True,False,False,False,False
Gujarat,True,False,True,False,True,True,True,True
Haryana,True,False,True,False,True,True,True,True
Himachal Pradesh,False,True,False,True,False,False,False,False


---
## Section C — Frequent Itemset Generation (Apriori)

### Support Metric in Small Sample ($n = 36$)
Support measures the proportion of all 36 States/UTs that contain the given itemset:
$$\text{Support}(X) = \frac{\sigma(X)}{N} = \frac{\text{Number of States with } X}{36}$$

With $n = 36$:
- $1\text{ state} = 2.78\%$
- $4\text{ states} = 11.11\%$
- $9\text{ states} = 25.00\%$
- $18\text{ states} = 50.00\%$

We establish a minimum support threshold of **$\text{min\_support} = 0.25$ (at least 9 States/UTs)** to identify regional profile associations while filtering out sparse combinations.


In [4]:
# Mine frequent itemsets using Apriori
min_supp = 0.25
frequent_itemsets = mine_frequent_itemsets(transaction_matrix, min_support=min_supp)

print(f"Total Frequent Itemsets Discovered (min_support >= {min_supp:.2f} / 9 States): {len(frequent_itemsets)}")
print(f"Breakdown by Itemset Size:")
print(frequent_itemsets['itemset_size'].value_counts().sort_index().to_dict())

print("\nTop 15 Frequent Itemsets by Support:")
display(frequent_itemsets[['itemset_str', 'itemset_size', 'support', 'support_count']].head(15))


Total Frequent Itemsets Discovered (min_support >= 0.25 / 9 States): 129
Breakdown by Itemset Size:
{1: 8, 2: 22, 3: 35, 4: 35, 5: 21, 6: 7, 7: 1}

Top 15 Frequent Itemsets by Support:


,itemset_str,itemset_size,support,support_count
0,HIGH_IDENTITY_THEFT,1,0.527778,19
1,HIGH_FRAUD_MOTIVE,1,0.500000,18
2,HIGH_SEC66D_CHEATING,1,0.500000,18
3,HIGH_IT_ACT_SHARE,1,0.500000,18
4,HIGH_EXTORTION_MOTIVE,1,0.500000,18
5,HIGH_SEXUAL_EXPLOITATION_MOTIVE,1,0.500000,18
6,HIGH_WOMEN_CYBERCRIME,1,0.500000,18
7,HIGH_CHILD_CYBERCRIME,1,0.500000,18
8,HIGH_EXTORTION_MOTIVE + HIGH_IDENTITY_THEFT,2,0.472222,17
9,HIGH_EXTORTION_MOTIVE + HIGH_SEXUAL_EXPLOITATION_MOTIVE,2,0.472222,17


In [5]:
# Visualize Top Frequent Itemsets
fig_itemsets = plot_frequent_itemset_support(
    frequent_itemsets,
    top_n=15,
    save_path=str(project_root / 'outputs' / 'figures' / '13_apriori_itemset_support.png')
)
plt.show()


<string>:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section D — Association Rule Generation & Filtering

### Evaluation Metrics:
1. **Support**: $\text{Support}(A \to B) = P(A \cap B) = \frac{\text{States with both } A \text{ and } B}{36}$
2. **Confidence**: $\text{Confidence}(A \to B) = P(B \mid A) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)}$
3. **Lift**: $\text{Lift}(A \to B) = \frac{P(B \mid A)}{P(B)} = \frac{\text{Confidence}(A \to B)}{\text{Support}(B)}$
   - $\text{Lift} > 1.0$: Positive state profile association (co-occur more than random baseline).
   - $\text{Lift} \approx 1.0$: Statistical independence across state profiles.
   - $\text{Lift} < 1.0$: Negative co-occurrence (characteristic of divergent state clusters).

### Filtering Criteria:
- Minimum Support: **$0.25$** ($k \ge 9$ States/UTs)
- Minimum Confidence: **$0.60$** ($60\%$)
- Minimum Lift: **$> 1.0$** (strictly positive association)


In [6]:
# Mine and filter association rules
min_conf = 0.60
min_lift = 1.0

rules = mine_association_rules(
    frequent_itemsets,
    n_transactions=len(transaction_matrix),
    min_confidence=min_conf,
    min_lift=min_lift
)

print(f"Total Filtered Association Rules (min_conf >= {min_conf}, lift > {min_lift}): {len(rules)}")

# Separate 1-to-1 pair rules for primary interpretation
rules['ant_len'] = rules['antecedent_str'].apply(lambda s: len(s.split(' + ')))
rules['con_len'] = rules['consequent_str'].apply(lambda s: len(s.split(' + ')))

pair_rules = rules[(rules['ant_len'] == 1) & (rules['con_len'] == 1)].sort_values(
    by=['lift', 'confidence', 'support'], ascending=[False, False, False]
).reset_index(drop=True)

print(f"\nPair Rules (1 Antecedent -> 1 Consequent): {len(pair_rules)} (used for human-readable profile analysis)")
display(pair_rules[['rule_str', 'support', 'support_count', 'confidence', 'lift']].head(15))


Total Filtered Association Rules (min_conf >= 0.6, lift > 1.0): 1924

Pair Rules (1 Antecedent -> 1 Consequent): 42 (used for human-readable profile analysis)


,rule_str,support,support_count,confidence,lift
0,HIGH_SEXUAL_EXPLOITATION_MOTIVE -> HIGH_EXTORTION_MOTIVE,0.472222,17,0.944444,1.888889
1,HIGH_EXTORTION_MOTIVE -> HIGH_SEXUAL_EXPLOITATION_MOTIVE,0.472222,17,0.944444,1.888889
2,HIGH_EXTORTION_MOTIVE -> HIGH_IDENTITY_THEFT,0.472222,17,0.944444,1.789474
3,HIGH_IDENTITY_THEFT -> HIGH_EXTORTION_MOTIVE,0.472222,17,0.894737,1.789474
4,HIGH_FRAUD_MOTIVE -> HIGH_EXTORTION_MOTIVE,0.444444,16,0.888889,1.777778
5,HIGH_EXTORTION_MOTIVE -> HIGH_FRAUD_MOTIVE,0.444444,16,0.888889,1.777778
6,HIGH_WOMEN_CYBERCRIME -> HIGH_SEXUAL_EXPLOITATION_MOTIVE,0.444444,16,0.888889,1.777778
7,HIGH_SEXUAL_EXPLOITATION_MOTIVE -> HIGH_WOMEN_CYBERCRIME,0.444444,16,0.888889,1.777778
8,HIGH_CHILD_CYBERCRIME -> HIGH_SEXUAL_EXPLOITATION_MOTIVE,0.444444,16,0.888889,1.777778
9,HIGH_SEXUAL_EXPLOITATION_MOTIVE -> HIGH_CHILD_CYBERCRIME,0.444444,16,0.888889,1.777778


In [7]:
# Visualize Association Rules Scatter Plot
fig_scatter = plot_association_rules_scatter(
    rules,
    save_path=str(project_root / 'outputs' / 'figures' / '14_association_rules_scatter.png')
)
plt.show()


<string>:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


---
## Section E — Substantive Profile Interpretation

### Analytical Question 1: Do states with High Fraud Motive frequently exhibit High Sec. 66D Activity?
- **Forward Rule**: `HIGH_FRAUD_MOTIVE -> HIGH_SEC66D_CHEATING`
  - Support: **$41.67\%$** ($15 / 36$ State/UT profiles)
  - Confidence: **$83.33\%$** ($15 / 18$ states with high fraud motive)
  - Lift: **$1.67$** relative to independence baseline
- **Reverse Rule**: `HIGH_SEC66D_CHEATING -> HIGH_FRAUD_MOTIVE`
  - Support: **$41.67\%$** ($15 / 36$ State/UT profiles)
  - Confidence: **$83.33\%$** ($15 / 18$ states with high Sec. 66D activity)
  - Lift: **$1.67$**
- **Finding**: High fraud motive is associated with high Sec. 66D personation cheating across 15 of 36 State/UT profiles, showing a confidence of $83.33\%$ and a lift of $1.67$.

---

### Analytical Question 2: Does High Identity Theft Co-Occur with High Fraud Motive?
- **Forward Rule**: `HIGH_IDENTITY_THEFT -> HIGH_FRAUD_MOTIVE`
  - Support: **$44.44\%$** ($16 / 36$ State/UT profiles)
  - Confidence: **$84.21\%$** ($16 / 19$ states with high identity theft)
  - Lift: **$1.68$**
- **Reverse Rule**: `HIGH_FRAUD_MOTIVE -> HIGH_IDENTITY_THEFT`
  - Support: **$44.44\%$** ($16 / 36$ State/UT profiles)
  - Confidence: **$88.89\%$** ($16 / 18$ states with high fraud motive)
  - Lift: **$1.68$**
- **Finding**: States with above-median identity theft (Sec. 66C) co-occur with high fraud motive in 16 of 36 State/UT profiles (confidence $84.21\%$, lift $1.68$).

---

### Analytical Question 3: Does High IT Act Share Co-Occur with Specific Motive Profiles?
- **Finding**: Under the selected median thresholds, `HIGH_IT_ACT_SHARE` does not produce a positive-lift association ($\text{Lift} > 1.0$) with the examined raw motive-count indicators.
- **Explanation**: In the observed transaction matrix, states with above-median IT Act shares ($>68.75\%$) do not exhibit above-median counts for raw fraud, extortion, or sexual exploitation motives at rates exceeding the random baseline (Lift $\le 1.0$).

---

### Analytical Question 4: Are Women and Children Subset Concentrations Associated with Specific Profiles?
- **Forward Rule**: `HIGH_WOMEN_CYBERCRIME -> HIGH_SEXUAL_EXPLOITATION_MOTIVE`
  - Support: **$44.44\%$** ($16 / 36$ State/UT profiles)
  - Confidence: **$88.89\%$** ($16 / 18$ states)
  - Lift: **$1.78$**
- **Reverse Rule**: `HIGH_SEXUAL_EXPLOITATION_MOTIVE -> HIGH_WOMEN_CYBERCRIME`
  - Support: **$44.44\%$** ($16 / 36$ State/UT profiles)
  - Confidence: **$88.89\%$** ($16 / 18$ states)
  - Lift: **$1.78$**
- **Subset Co-occurrence**: `HIGH_WOMEN_CYBERCRIME -> HIGH_CHILD_CYBERCRIME`
  - Support: **$44.44\%$** ($16 / 36$ State/UT profiles)
  - Confidence: **$88.89\%$** ($16 / 18$ states)
  - Lift: **$1.78$**
- **Finding**: Above-median cybercrimes against women co-occur with above-median cybercrimes against children and above-median sexual exploitation motives across 16 of 36 State/UT profiles (confidence $88.89\%$, lift $1.78$).


In [8]:
# Export Tables and Verify Output Files
exported_paths = export_association_tables(
    transaction_matrix=transaction_matrix,
    threshold_summary=threshold_summary,
    frequent_itemsets=frequent_itemsets,
    rules=rules,
    output_dir=str(project_root / 'outputs' / 'tables')
)

print("Exported Association Rule Mining Tables:")
for k, v in exported_paths.items():
    p = Path(v)
    print(f"  - {k:<20}: {p.name} ({p.stat().st_size:,} bytes)")


Exported Association Rule Mining Tables:
  - transaction_matrix  : state_transaction_matrix.csv (2,222 bytes)
  - item_thresholds     : association_item_thresholds.csv (1,497 bytes)
  - frequent_itemsets   : frequent_itemsets.csv (13,902 bytes)
  - association_rules   : association_rules.csv (833,783 bytes)


---
## Section F — Methodological Limitations & Verification

### Explicit Analytical Boundaries:
1. **Small Sample Size ($n = 36$)**: Support is discretized in increments of $1/36 \approx 2.78\%$. Small shifts in state classification can alter support counts.
2. **Threshold Sensitivity**: The median split was selected for mathematical symmetry ($50\%$ marginal support per item). Alternative cutoffs alter itemset density.
3. **No Incident-Level Inference**: A rule such as $\text{Fraud} \to \text{Identity Theft}$ indicates that State/UT profiles with above-median fraud also tend to have above-median identity theft; it does **not** prove that individual fraud incidents involve identity theft.
4. **Non-Causal Nature**: These patterns describe statistical co-occurrence among the selected State/UT-level indicators in the observed dataset. The analysis does not test causal mechanisms or external explanatory factors.
